# Programming Assignment: Inconsistency Resolution

Welcome to the **Inconsistency Resolution** assignment!

Real-world data is messy — the same entity can be stored in dozens of different ways. Consider these common patterns:

| Column Type | Example Inconsistencies |
|-------------|------------------------|
| **Country names** | `'USA'`, `'US'`, `'U.S.A'`, `'United States'`, `'united states of america'` |
| **Currency values** | `'$1,200.50'`, `'900.00 USD'`, `'1200'`, `'€800'` |
| **Date formats** | `'2023-01-15'`, `'01/15/2023'`, `'January 15, 2023'`, `'15-Jan-23'` |
| **Weight / units** | `'72.5kg'`, `'80 KG'`, `'65'`, `'90.1 Kg'` |
| **Categorical text** | `'Active'`, `'ACTIVE'`, `'active'`, `'aCtIvE'` |

Without resolving these inconsistencies, your analyses will silently produce wrong results — group-bys will fragment, merges will miss matches, and models will treat equivalent values as distinct categories.

**What You Will Do in This Assignment:**

* Implement **country name standardisation** to unify all USA variants
* Implement **currency parsing** to extract numeric values from messy strings
* Implement **date format standardisation** to produce uniform `YYYY-MM-DD` output
* Implement **weight parsing** to strip unit suffixes and convert to float
* Implement **text case standardisation** to collapse mixed-case category labels

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

---

## Table of Contents
- [Imports](#imports)
- [1 - Understanding the Data](#1---understanding-the-data)
- [2 - Country Name Standardisation](#2---country-name-standardisation)
    - **[Exercise 1 - `standardize_country`](#exercise-1---standardize_country)**
- [3 - Currency Parsing](#3---currency-parsing)
    - **[Exercise 2 - `parse_currency`](#exercise-2---parse_currency)**
- [4 - Date Format Standardisation](#4---date-format-standardisation)
    - **[Exercise 3 - `standardize_date_format`](#exercise-3---standardize_date_format)**
- [5 - Weight Parsing](#5---weight-parsing)
    - **[Exercise 4 - `parse_weight`](#exercise-4---parse_weight)**
- [6 - Text Case Standardisation](#6---text-case-standardisation)
    - **[Exercise 5 - `standardize_text_case`](#exercise-5---standardize_text_case)**
- [7 - Visualizing Results](#7---visualizing-results)

<a name='imports'></a>
## Imports

In [ ]:
import re
import numpy as np
import pandas as pd

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-the-data'></a>
## 1 - Understanding the Data

We generate a synthetic dataset with **5 columns**, each exhibiting a different type of inconsistency:

| Column | Inconsistency Type |
|--------|-------------------|
| `country` | Mixed spellings and abbreviations for country names |
| `price` | Currency strings with symbols, commas, and suffixes |
| `date` | Date strings in multiple formats |
| `weight_kg` | Numeric values with varying unit suffixes |
| `status` | Categorical labels with inconsistent casing |

Run the cells below to explore the raw data before you start cleaning it.

In [ ]:
df = helper_utils.generate_messy_dataset(n_samples=200, random_state=42)
print(f"Shape: {df.shape}")
print(f"\nUnique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique — sample: {df[col].unique()[:5].tolist()}")

In [ ]:
helper_utils.plot_inconsistency_summary(df)

<a name='2---country-name-standardisation'></a>
## 2 - Country Name Standardisation

<a name='exercise-1---standardize_country'></a>
### **Exercise 1 - `standardize_country`**

**Your Task:**
Implement a function that maps all recognised USA name variants to the canonical string `'United States'`.

**Requirements:**
- Accept a `pd.Series` of country name strings.
- Return a new `pd.Series` where every USA variant is replaced with `'United States'`.
- Non-USA values should be returned unchanged.
- Matching must be **case-insensitive** and strip leading/trailing whitespace.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Define a set of lowercase USA variants: `{'usa', 'us', 'u.s.a', 'u.s.', 'united states', 'united states of america', ...}`.
2. Use `series.apply(lambda x: ...)` to process each value.
3. Inside the lambda, normalise the value: `str(x).lower().strip()`.
4. Check membership in your variant set — if it matches, return `'United States'`; otherwise return `x`.
5. Additionally normalise by removing dots and spaces to catch variants like `'U.S.A.'` → `'usa'`.

</details>

In [ ]:
# GRADED FUNCTION: standardize_country
def standardize_country(series):
    """
    Standardize country name variants to a canonical form.

    Args:
        series (pd.Series): Series with country name strings.

    Returns:
        pd.Series: Series with standardized country names.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
country_clean = standardize_country(df['country'])
print(f"Before unique values: {df['country'].unique().tolist()}")
print(f"After unique values:  {country_clean.unique().tolist()}")

In [ ]:
unittests.exercise_1(standardize_country)

<a name='3---currency-parsing'></a>
## 3 - Currency Parsing

<a name='exercise-2---parse_currency'></a>
### **Exercise 2 - `parse_currency`**

**Your Task:**
Implement a function that extracts the numeric value from messy currency strings such as `'$1,200.50'`, `'900.00 USD'`, or `'€1500'`.

**Requirements:**
- Accept a `pd.Series` of currency strings.
- Return a `pd.Series` of `float` values.
- Values that cannot be parsed should become `NaN`.
- Handle `NaN` inputs gracefully (pass them through as `NaN`).

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Define an inner helper `_parse(val)` that processes a single string.
2. Return `np.nan` early if `pd.isna(val)`.
3. Use `re.sub(r'[^\d.]', '', str(val))` to strip everything except digits and dots.
4. Try `float(cleaned)` — if `cleaned` is empty or conversion fails, return `np.nan`.
5. Apply the helper with `series.apply(_parse)`.

</details>

In [ ]:
# GRADED FUNCTION: parse_currency
def parse_currency(series):
    """
    Extract numeric values from currency strings.

    Args:
        series (pd.Series): Series with currency strings (e.g. '$1,200.50', '900 USD').

    Returns:
        pd.Series: Float Series with extracted values; unparseable values become NaN.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
price_clean = parse_currency(df['price'])
print(f"Before sample: {df['price'].head(3).tolist()}")
print(f"After sample:  {price_clean.head(3).tolist()}")
print(f"dtype: {price_clean.dtype}")

In [ ]:
unittests.exercise_2(parse_currency)

<a name='4---date-format-standardisation'></a>
## 4 - Date Format Standardisation

<a name='exercise-3---standardize_date_format'></a>
### **Exercise 3 - `standardize_date_format`**

**Your Task:**
Parse date strings that appear in multiple formats and output them uniformly as `'YYYY-MM-DD'` strings.

**Requirements:**
- Accept a `pd.Series` of mixed-format date strings.
- Accept an `output_format` parameter (default `'%Y-%m-%d'`).
- Return a `pd.Series` of strings formatted according to `output_format`.
- Dates that cannot be parsed should become `NaN` (not raise an exception).

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Use `pd.to_datetime(series, format='mixed', errors='coerce')` — `format='mixed'` (pandas ≥ 2.0) infers the format independently for each value, handling a series that mixes e.g. `'2023-01-15'` and `'01/15/2023'`.
2. The result is a datetime Series; invalid entries become `NaT`.
3. Call `.dt.strftime(output_format)` to convert valid dates to strings.
4. Use `.where(parsed.notna(), other=np.nan)` to replace `NaT`-derived positions with actual `NaN`.

</details>

In [ ]:
# GRADED FUNCTION: standardize_date_format
def standardize_date_format(series, output_format='%Y-%m-%d'):
    """
    Parse date strings in various formats and output them uniformly.

    Args:
        series (pd.Series): Series with date strings in mixed formats.
        output_format (str): Target strftime format (default '%Y-%m-%d').

    Returns:
        pd.Series: String Series in output_format; invalid dates become NaN.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
date_clean = standardize_date_format(df['date'])
print(f"Before sample: {df['date'].head(3).tolist()}")
print(f"After sample:  {date_clean.head(3).tolist()}")

In [ ]:
unittests.exercise_3(standardize_date_format)

<a name='5---weight-parsing'></a>
## 5 - Weight Parsing

<a name='exercise-4---parse_weight'></a>
### **Exercise 4 - `parse_weight`**

**Your Task:**
Parse weight strings by stripping unit suffixes (e.g. `'kg'`, `'KG'`, `'Kg'`) and converting the remaining numeric part to `float`.

**Requirements:**
- Accept a `pd.Series` of weight strings such as `'72.5kg'`, `'80 KG'`, `'65'`.
- Return a `pd.Series` of `float` values.
- Empty strings and unparseable values should become `NaN`.
- Handle `NaN` inputs gracefully.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Define an inner helper `_parse(val)`.
2. Return `np.nan` early if `pd.isna(val)`.
3. Use `re.sub(r'[a-zA-Z\s]', '', str(val)).strip()` to remove all letters and whitespace — this leaves only the numeric part.
4. Try `float(cleaned)` — if `cleaned` is empty or conversion fails, return `np.nan`.
5. Apply with `series.apply(_parse)`.

</details>

In [ ]:
# GRADED FUNCTION: parse_weight
def parse_weight(series):
    """
    Parse weight values by stripping unit suffixes and converting to float.

    Args:
        series (pd.Series): Series with weight strings e.g. '72.5kg', '80 KG', '65'.

    Returns:
        pd.Series: Float Series; empty strings or unparseable values become NaN.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
weight_clean = parse_weight(df['weight_kg'])
print(f"Before sample: {df['weight_kg'].head(3).tolist()}")
print(f"After sample:  {weight_clean.head(3).tolist()}")

In [ ]:
unittests.exercise_4(parse_weight)

<a name='6---text-case-standardisation'></a>
## 6 - Text Case Standardisation

<a name='exercise-5---standardize_text_case'></a>
### **Exercise 5 - `standardize_text_case`**

**Your Task:**
Standardise the casing of all string values in a Series to a uniform format — useful for collapsing `'Active'`, `'ACTIVE'`, and `'active'` into a single canonical label.

**Requirements:**
- Accept a `pd.Series` of string values and a `case` parameter (`'lower'`, `'upper'`, or `'title'`).
- Return a new `pd.Series` with all values converted to the specified case.
- If an unrecognised `case` value is passed, return a copy of the original Series unchanged.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Use `series.str.lower()` for lowercase, `series.str.upper()` for uppercase, and `series.str.title()` for title case.
2. A simple `if/elif/elif/else` chain based on the `case` parameter is the cleanest approach.
3. For the fallback, return `series.copy()` to avoid mutating the original.

</details>

In [ ]:
# GRADED FUNCTION: standardize_text_case
def standardize_text_case(series, case='lower'):
    """
    Standardize string casing across a Series.

    Args:
        series (pd.Series): Series with string values.
        case (str): Target case: 'lower', 'upper', or 'title'.

    Returns:
        pd.Series: Series with uniform casing.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
status_clean = standardize_text_case(df['status'], case='lower')
print(f"Before unique: {sorted(df['status'].unique())}")
print(f"After unique:  {sorted(status_clean.unique())}")

In [ ]:
unittests.exercise_5(standardize_text_case)

## 7 - Visualizing Results

In [ ]:
helper_utils.visualize_before_after(df['country'], standardize_country(df['country']), 'country')
helper_utils.visualize_before_after(df['status'], standardize_text_case(df['status']), 'status')